# Silver Layer — ERP: Customer AZ12

Transforms the bronze `workspace.bronze.cust_az12` table into `workspace.silver.customer_info1` with meaningful, readable column names and cleaned data.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (`CID` -> `customer_id`, `BDATE` -> `birth_date`, `GEN` -> `gender`) |
| 2 | **Clean Customer ID** | Trimmed whitespace and removed `NAS` prefix from `customer_id` (e.g., `NASAW00011000` -> `AW00011000`) |
| 3 | **Gender Standardization** | Trimmed and mapped all gender variants to consistent values: `M`/`M ` -> `Male`, `F`/`F ` -> `Female`, `NULL`/spaces/empty -> `Unknown` |

---

## Column Mapping (Bronze to Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `CID` | `customer_id` | Renamed + Trimmed + `NAS` prefix removed |
| `BDATE` | `birth_date` | Renamed |
| `GEN` | `gender` | Renamed + Trimmed + Standardized |

---

## Gender Value Mapping

| Bronze Value(s) | Silver Value | Count |
|---|---|---|
| `Male` | `Male` | 8,608 |
| `Female` | `Female` | 8,391 |
| `NULL`, `  `, empty | `Unknown` | 1,476 |
| `M`, `M ` | `Male` | 4 |
| `F`, `F ` | `Female` | 5 |

---

## Implementation

All transformations are implemented in **PySpark** (`pyspark.sql.functions`):
- `col()` — select and alias columns
- `trim()` — remove leading/trailing whitespace
- `when()` + `otherwise()` — conditional mapping for gender standardization
- `df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable()` — write to silver layer

In [0]:
from pyspark.sql.functions import col, trim, when, lit, regexp_replace

df = (spark.read.table("workspace.bronze.cust_az12")
    .select(
        regexp_replace(trim(col("CID")), "^NAS", "").alias("customer_id"),
        col("BDATE").alias("birth_date"),
        when(trim(col("GEN")) == "Male", "Male")
            .when(trim(col("GEN")) == "M", "Male")
            .when(trim(col("GEN")) == "Female", "Female")
            .when(trim(col("GEN")) == "F", "Female")
            .otherwise("Unknown")
            .alias("gender"),
    )
)

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.customer_info1")

In [0]:
display(spark.read.table("workspace.silver.customer_info1").limit(10))